In [1]:
import os, sys
from pathlib import Path
PROJECT = Path(r"C:\Users\hp\Documents\teliport_prototype")
os.chdir(PROJECT / "notebooks")
sys.path.insert(0, str(PROJECT / "src")); sys.path.insert(0, str(PROJECT))
print("Working dir:", Path.cwd())

Working dir: C:\Users\hp\Documents\teliport_prototype\notebooks


In [2]:
feat_file = PROJECT / "src" / "features.py"
s = feat_file.read_text(encoding="utf-8")
if ".bfill()" in s:
    lines = s.split("\n")
    i = next(k for k, line in enumerate(lines) if ".bfill()" in line)
    indent = lines[i][: len(lines[i]) - len(lines[i].lstrip())]
    lines[i] = indent + "# no back-fill: an early gap is never filled with later (future) readings -> column median below"
    s = "\n".join(lines)
    feat_file.write_text(s, encoding="utf-8")
    print("src/features.py fixed: back-fill removed")
else:
    print("src/features.py already fixed")

src/features.py fixed: back-fill removed


In [3]:
agent_file = PROJECT / "src" / "triage_agent.py"
s = agent_file.read_text(encoding="utf-8")

helpers = '''
ACTIVE_STATUSES = {"AUTO_APPROVED", "PENDING_HUMAN_REVIEW", "APPROVED_BY_HUMAN"}


def parse_top_sensors(raw):
    """Accept a list of (sensor, name, pct) tuples, a list of dicts, or the same as a JSON/CSV string."""
    if isinstance(raw, str):
        try:
            raw = json.loads(raw)
        except ValueError:
            try:
                raw = ast.literal_eval(raw)
            except (ValueError, SyntaxError):
                raw = []
    out = []
    for item in raw if isinstance(raw, (list, tuple)) else []:
        if isinstance(item, dict):
            item = (item.get("sensor"), item.get("name", item.get("sensor")), item.get("share_%", 0))
        if isinstance(item, (list, tuple)) and len(item) == 3:
            sensor, name, pct = item
            try:
                pct = float(pct)
            except (TypeError, ValueError):
                continue
            if sensor in SENSOR_COMPONENT and 0 <= pct <= 100:
                out.append((sensor, str(name), pct))
    return out


class ActiveOrderRegistry:
    """One active work order per engine. Re-scoring the same engine UPDATES its order instead of
    creating a duplicate: severity can only go up and the due cycle can only move earlier."""

    def __init__(self, path):
        self.path = Path(path)
        self.orders = json.loads(self.path.read_text()) if self.path.exists() else {}

    def register(self, wo):
        cur = self.orders.get(wo["asset"])
        if cur is None or cur.get("status") not in ACTIVE_STATUSES:
            wo = dict(wo, history=[{"cycle": wo["cycles_seen"], "severity": wo["severity"]}])
            self.orders[wo["asset"]] = wo
            return wo, True
        if SEVERITY_ORDER[wo["severity"]] > SEVERITY_ORDER[cur["severity"]]:
            for k in ("severity", "recommended_action", "rules_fired", "suspected_component", "top_sensors"):
                cur[k] = wo[k]
        if wo["status"] == "PENDING_HUMAN_REVIEW" and cur["status"] == "AUTO_APPROVED":
            cur["status"], cur["review_reasons"] = "PENDING_HUMAN_REVIEW", wo["review_reasons"]
        cur["due_by_cycle"] = min(cur["due_by_cycle"], wo["due_by_cycle"])
        cur["due_within_cycles"] = max(0, cur["due_by_cycle"] - wo["cycles_seen"])
        for k in ("cycles_seen", "pred_RUL", "rul_lower", "rul_upper", "decision_RUL", "anomaly_score",
                  "alarm", "model_version"):
            cur[k] = wo[k]
        cur["last_updated"] = wo["created_at"]
        entry = {"cycle": wo["cycles_seen"], "severity": wo["severity"]}
        history = cur.setdefault("history", [])
        if not history or history[-1] != entry:
            history.append(entry)
        return cur, False

    def save(self):
        self.path.parent.mkdir(parents=True, exist_ok=True)
        tmp = self.path.with_suffix(".tmp")
        tmp.write_text(json.dumps(self.orders, indent=2))
        tmp.replace(self.path)


'''

new_run = '''    # ---------- 3. run over a fleet snapshot ----------
    def _write(self, wo):
        self.out_dir.mkdir(parents=True, exist_ok=True)
        target = self.out_dir / f"{wo['work_order_id']}.json"
        tmp = target.with_suffix(".tmp")
        tmp.write_text(json.dumps(wo, indent=2))
        tmp.replace(target)

    def run(self, snapshots: pd.DataFrame) -> pd.DataFrame:
        registry = ActiveOrderRegistry(self.out_dir / "active_registry.json") if self.out_dir else None
        rows, new_orders = [], []
        created = updated = 0
        for snap in snapshots.to_dict("records"):
            d = self.assess(snap)
            wo = self.create_work_order(d, write=False)
            if wo is not None and registry is not None:
                wo, fresh = registry.register(wo)
                self._write(wo)
                created, updated = created + int(fresh), updated + int(not fresh)
                if fresh:
                    new_orders.append(wo)
            rows.append({"subset": d.subset, "unit": d.unit, "cycles_seen": d.cycles_seen,
                         "pred_RUL": d.pred_RUL, "rul_lower": d.rul_lower, "rul_upper": d.rul_upper,
                         "anomaly_score": d.anomaly_score, "alarm": d.alarm,
                         "severity": d.severity, "suspected_component": d.suspected_component,
                         "needs_human_review": d.needs_human_review,
                         "work_order_id": wo["work_order_id"] if wo else None,
                         "wo_status": wo["status"] if wo else "NO_ACTION",
                         "due_within_cycles": wo["due_within_cycles"] if wo else None})
        self.last_run_summary = {"new_orders": created, "updated_existing_orders": updated}
        if registry is not None:
            registry.save()
            if new_orders:
                log = pd.DataFrame([{k: (json.dumps(v) if isinstance(v, (list, dict)) else v)
                                     for k, v in o.items()} for o in new_orders])
                log_path = self.out_dir / "work_order_log.csv"
                log.to_csv(log_path, mode="a", header=not log_path.exists(), index=False)
        return pd.DataFrame(rows)
'''

if "class ActiveOrderRegistry" not in s:
    assert "import json\n" in s and "class TriageAgent:" in s and "    # ---------- 3. run over a fleet snapshot ----------" in s
    s = s.replace("import json\n", "import ast\nimport json\n", 1)
    s = s.replace("class TriageAgent:", helpers.lstrip("\n") + "class TriageAgent:", 1)
    s = s.replace('        top = list(snap.get("top_sensors") or [])',
                  '        top = parse_top_sensors(snap.get("top_sensors") or [])', 1)
    old_def = "    def create_work_order(self, d: Decision, now: datetime | None = None) -> dict | None:"
    assert old_def in s
    s = s.replace(old_def, "    def create_work_order(self, d: Decision, now: datetime | None = None,\n"
                           "                          write: bool = True) -> dict | None:", 1)
    old_w = "        if self.out_dir:\n            self.out_dir.mkdir(parents=True, exist_ok=True)\n"
    assert old_w in s
    s = s.replace(old_w, "        if self.out_dir and write:\n            self.out_dir.mkdir(parents=True, exist_ok=True)\n", 1)
    assert '            "due_within_cycles": due,\n' in s
    s = s.replace('            "due_within_cycles": due,\n',
                  '            "due_within_cycles": due,\n            "due_by_cycle": d.cycles_seen + due,\n', 1)
    head = s[: s.index("    # ---------- 3. run over a fleet snapshot ----------")]
    s = head + new_run
    agent_file.write_text(s, encoding="utf-8")
    print("src/triage_agent.py updated: safe top_sensors parsing + one active work order per engine")
else:
    print("src/triage_agent.py already updated")

src/triage_agent.py updated: safe top_sensors parsing + one active work order per engine


In [5]:
%%writefile ../src/sensor_health.py
"""Sensor-fault detection by analytical redundancy ("virtual sensors").

When an engine wears, its sensors drift TOGETHER in a physically consistent way. When one SENSOR
breaks (sudden bias, stuck value, drift), it stops agreeing with the other 13. For every sensor we
learn a small model that predicts it from the other sensors; a persistent, large disagreement means
that sensor (not the engine) is suspect. The suspect sensor can then be replaced by its virtual
value to give a backup RUL. Nothing is corrected silently: the agent only uses the backup to
ESCALATE (never to downgrade) and always sends the engine to human review.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold


class SensorHealth:
    def __init__(self, quantile=0.9995, persistence=5, alpha=1.0, n_splits=3):
        self.quantile = quantile
        self.persistence = persistence
        self.alpha = alpha
        self.n_splits = n_splits

    # ---------- fit on TRAINING engines only ----------
    def fit(self, feats, prep):
        self.sensors_ = list(prep.sensors)
        S = self.sensors_
        units = feats["unit"].to_numpy()
        oof = np.zeros((len(feats), len(S)))
        self.slow_, self.fast_, self.sigma_ = {}, {}, {}
        for j, s in enumerate(S):
            others = [o for o in S if o != s]
            Xe, ye = feats[[f"{o}_ewm" for o in others]].to_numpy(), feats[f"{s}_ewm"].to_numpy()
            for a, b in GroupKFold(n_splits=self.n_splits).split(Xe, groups=units):     # unseen-engine residuals
                oof[b, j] = ye[b] - Ridge(alpha=self.alpha).fit(Xe[a], ye[a]).predict(Xe[b])
            self.slow_[s] = Ridge(alpha=self.alpha).fit(Xe, ye)                        # detector (smoothed)
            Xz, yz = feats[[f"{o}_z" for o in others]].to_numpy(), feats[f"{s}_z"].to_numpy()
            self.fast_[s] = Ridge(alpha=self.alpha).fit(Xz, yz)                        # virtual sensor (per cycle)
            self.sigma_[s] = float(oof[:, j].std() + 1e-6)
        z = np.abs(oof / np.array([self.sigma_[s] for s in S]))
        self.threshold_ = float(np.quantile(z.max(axis=1), self.quantile))
        return self

    # ---------- detection ----------
    def scores(self, feats):
        S = self.sensors_
        out = {}
        for s in S:
            others = [o for o in S if o != s]
            pred = self.slow_[s].predict(feats[[f"{o}_ewm" for o in others]].to_numpy())
            out[s] = (feats[f"{s}_ewm"].to_numpy() - pred) / self.sigma_[s]
        return pd.DataFrame(out, index=feats.index)

    def last_suspects(self, feats):
        """{unit: (sensor, score)} for engines whose worst sensor disagreed for `persistence` cycles in a row."""
        sc = self.scores(feats).abs()
        worst = sc.max(axis=1)
        over = (worst > self.threshold_).astype(int)
        streak = over.groupby(feats["unit"]).rolling(self.persistence, min_periods=self.persistence).sum()
        streak = streak.reset_index(level=0, drop=True).reindex(feats.index).fillna(0)
        flagged = streak >= self.persistence
        last_idx = feats.groupby("unit").tail(1).index
        res = {}
        for i in last_idx:
            if flagged.loc[i]:
                s = sc.loc[i].idxmax()
                res[int(feats.loc[i, "unit"])] = (s, round(float(sc.loc[i, s]), 1))
        return res

    # ---------- backup input: replace the suspect sensor by its virtual value ----------
    def repair_raw(self, raw, feats, prep, unit, sensor):
        S = self.sensors_
        others = [o for o in S if o != sensor]
        f = feats[feats["unit"] == unit]
        zhat = self.fast_[sensor].predict(f[[f"{o}_z" for o in others]].to_numpy())
        reg = f["regime"].to_numpy()
        virtual = zhat * prep.sd_.loc[reg, sensor].to_numpy() + prep.mu_.loc[reg, sensor].to_numpy()
        vmap = pd.Series(virtual, index=f["cycle"].astype(int).to_numpy())
        eng = raw[raw["unit"] == unit].copy()
        eng[sensor] = eng["cycle"].astype(int).map(vmap).to_numpy()
        return eng

Writing ../src/sensor_health.py


In [6]:
def patch(path, fixes, done_marker):
    s = path.read_text(encoding="utf-8")
    if done_marker in s:
        print(f"{path.name}: already updated")
        return
    for old, new in fixes:
        assert old in s, f"{path.name}: could not find -> {old[:70]}"
        s = s.replace(old, new, 1)
    path.write_text(s, encoding="utf-8")
    print(f"{path.name}: updated")

SRC = PROJECT / "src"

# ---- training.py: fit + save the sensor-health model with every pipeline ----
patch(SRC / "training.py", [
    ("from anomaly import HealthMonitor\n", "from anomaly import HealthMonitor\nfrom sensor_health import SensorHealth\n"),
    ("    hm = HealthMonitor(**monitor_cfg).fit(tr, prep.sensors)\n",
     "    hm = HealthMonitor(**monitor_cfg).fit(tr, prep.sensors)\n"
     "    sh = SensorHealth().fit(tr, prep)                                 # sensor-fault detector\n"),
    ('"params": params, "monitor_cfg": monitor_cfg, "train_features": tr}',
     '"params": params, "monitor_cfg": monitor_cfg, "train_features": tr, "sh": sh}'),
    ('    joblib.dump(p["hm"], models_dir / f"health_monitor_{fd}.joblib")\n',
     '    joblib.dump(p["hm"], models_dir / f"health_monitor_{fd}.joblib")\n'
     '    if p.get("sh") is not None:\n'
     '        joblib.dump(p["sh"], models_dir / f"sensor_health_{fd}.joblib")\n'),
], "SensorHealth")

# ---- inference.py: flag suspect sensors and compute a backup RUL without them ----
patch(SRC / "inference.py", [
    ('    snap["model_version"] = model_version\n    return snap',
     '    snap["model_version"] = model_version\n'
     '    snap["sensor_suspect"], snap["sensor_fault_score"] = "", 0.0\n'
     '    snap["rul_backup"], snap["rul_backup_lower"] = np.nan, np.nan\n'
     '    sh_path = Path(models_dir) / f"sensor_health_{fd}.joblib"\n'
     '    if sh_path.exists():\n'
     '        sh = joblib.load(sh_path)\n'
     '        suspects = sh.last_suspects(feats)\n'
     '        if suspects:\n'
     '            fixed = pd.concat([sh.repair_raw(raw_df, feats, prep, u, s) for u, (s, _) in suspects.items()])\n'
     '            last2 = last_cycle(prep.transform(fixed)).set_index("unit")\n'
     '            bp = np.clip(rul_model.predict(last2[prep.feature_names_]), 0, None)\n'
     '            bl = unc.interval(bp)[0] if unc is not None else bp\n'
     '            backup = pd.DataFrame({"bp": np.round(bp, 1), "bl": np.round(bl, 1)}, index=last2.index)\n'
     '            for u, (s, score) in suspects.items():\n'
     '                m = snap["unit"] == u\n'
     '                snap.loc[m, "sensor_suspect"], snap.loc[m, "sensor_fault_score"] = s, score\n'
     '                snap.loc[m, "rul_backup"], snap.loc[m, "rul_backup_lower"] = backup.loc[u, "bp"], backup.loc[u, "bl"]\n'
     '    return snap'),
], "sensor_health_")

# ---- triage_agent.py: suspect sensor -> human review; backup RUL may ESCALATE, never downgrade ----
patch(SRC / "triage_agent.py", [
    ('    model_version: str = "unknown"\n',
     '    model_version: str = "unknown"\n    sensor_suspect: str = ""\n    rul_backup: float | None = None\n'),
    ("        if alarm and point > c.disagreement_rul:\n",
     "        suspect = snap.get(\"sensor_suspect\") or \"\"\n"
     "        suspect = suspect if isinstance(suspect, str) else \"\"\n"
     "        backup = snap.get(\"rul_backup\")\n"
     "        backup = None if backup is None or pd.isna(backup) else float(backup)\n"
     "        if suspect:\n"
     "            reasons.append(f\"Sensor {suspect} disagrees with the other sensors: verify the sensor before acting\")\n"
     "            if backup is not None:\n"
     "                b_low = float(snap.get(\"rul_backup_lower\", backup))\n"
     "                b_rul = b_low if (c.conservative_mode == \"all\" or (c.conservative_mode == \"critical\" and b_low <= crit_limit)) else backup\n"
     "                alt = band(b_rul)\n"
     "                if SEVERITY_ORDER[alt] > SEVERITY_ORDER[sev]:\n"
     "                    rules.append(f\"S1 RUL without suspect sensor {suspect} is {backup:.0f}: escalate {sev} to {alt}\")\n"
     "                    sev, rul = alt, min(rul, b_rul)\n\n"
     "        if alarm and point > c.disagreement_rul:\n"),
    ('                        model_version=str(snap.get("model_version", "unknown")))',
     '                        model_version=str(snap.get("model_version", "unknown")),\n'
     '                        sensor_suspect=suspect, rul_backup=backup)'),
    ('                         "needs_human_review": d.needs_human_review,\n',
     '                         "needs_human_review": d.needs_human_review, "sensor_suspect": d.sensor_suspect,\n'),
], "sensor_suspect")

# ---- run_pipeline.py: report how many engines had a suspect sensor ----
patch(PROJECT / "run_pipeline.py", [
    ("    agent_eval = evaluate_agent(decisions, subsets)\n",
     "    agent_eval = evaluate_agent(decisions, subsets)\n"
     "    agent_eval[\"sensor_flags\"] = int((decisions[\"sensor_suspect\"].fillna(\"\") != \"\").sum())\n"),
], "sensor_flags")

training.py: updated
inference.py: updated
triage_agent.py: updated
run_pipeline.py: updated


In [7]:
app = PROJECT / "dashboard" / "app.py"
s = app.read_text(encoding="utf-8")
if "sensor_suspect" not in s:
    old1 = '"alarm", "suspected_component", "needs_human_review", "wo_status", "due_within_cycles"]]'
    old2 = '    m[4].metric("Suspected component", row["suspected_component"].split(" (")[0])\n'
    assert old1 in s and old2 in s
    s = s.replace(old1, '"alarm", "suspected_component", "sensor_suspect", "needs_human_review", "wo_status",\n'
                        '                  "due_within_cycles"]]', 1)
    s = s.replace(old2, old2 +
        '    if isinstance(row.get("sensor_suspect"), str) and row["sensor_suspect"]:\n'
        '        st.warning(f"Sensor {row[\'sensor_suspect\']} disagrees with the other sensors: verify the sensor "\n'
        '                   "before acting. The engine was sent to human review.")\n', 1)
    app.write_text(s, encoding="utf-8")
    print("dashboard/app.py updated")
else:
    print("dashboard/app.py already updated")

dashboard/app.py updated


In [2]:
import warnings, joblib
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from cmapss_data import load_subset
from mlops import split_gate
from inference import score_fleet
from triage_agent import TriageAgent, TriageConfig, SEVERITY_ORDER as SO

DATA_DIR, MODELS = PROJECT / "data" / "raw" / "CMAPSS", PROJECT / "models"
SUBSETS_EVAL = [fd for fd in ["FD001", "FD002", "FD003", "FD004"] if (MODELS / f"sensor_health_{fd}.joblib").exists()]

def gate_endpoints(fd, targets=(10, 20, 35, 60, 100), min_history=40):
    """Gate engines (never trained on) cut at known remaining lives -> realistic 'current' snapshots."""
    train = load_subset(fd, DATA_DIR)[0].drop(columns=["subset"])
    pool = train[train["unit"] <= 80] if fd == "FD001" else train         # same gate as the pipeline
    _, gate = split_gate(pool)
    parts, truth = [], []
    for u, e in gate.groupby("unit"):
        life = int(e["cycle"].max())
        for t in targets:
            if life - t < min_history:
                continue
            part = e[e["cycle"] <= life - t].copy()
            part["unit"] = u * 1000 + t
            parts.append(part)
            truth.append({"unit": u * 1000 + t, "true_RUL": t})
    return pd.concat(parts, ignore_index=True).drop(columns=["RUL"]), pd.DataFrame(truth)

def inject(raw, fd, kind, size=3.0, length=30, seed=0):
    """Break ONE random sensor per engine during its last `length` cycles (size in sensor std units)."""
    prep = joblib.load(MODELS / f"preprocessor_{fd}.joblib")
    out, rng, faulty = raw.copy(), np.random.default_rng(seed), {}
    out[prep.sensors] = out[prep.sensors].astype(float)
    reg = prep.regimes(out)
    for u, idx in out.groupby("unit").groups.items():
        s = str(rng.choice(prep.sensors))
        faulty[u] = s
        cyc = out.loc[idx, "cycle"].to_numpy()
        on = cyc > cyc.max() - length
        rows = np.asarray(idx)[on]
        sd = prep.sd_.loc[reg[out.index.get_indexer(rows)], s].to_numpy()
        if kind == "bias_up":
            out.loc[rows, s] += size * sd
        elif kind == "bias_down":
            out.loc[rows, s] -= size * sd
        elif kind == "drift":
            out.loc[rows, s] += size * sd * np.linspace(0, 1, len(rows))
        elif kind == "stuck":
            out.loc[rows, s] = out.loc[np.asarray(idx)[~on][-1], s]
    return out, faulty

def agent_view(snaps, use_sensor_info):
    s = snaps.copy()
    if not use_sensor_info:
        s["sensor_suspect"], s["rul_backup"], s["rul_backup_lower"] = "", np.nan, np.nan
    return TriageAgent(TriageConfig()).run(s)

rows = []
for kind in ["clean", "bias_up", "bias_down", "stuck", "drift"]:
    acc = []
    for fd in SUBSETS_EVAL:
        raw, truth = gate_endpoints(fd)
        raw_f, faulty = (raw, {}) if kind == "clean" else inject(raw, fd, kind, seed=SUBSETS_EVAL.index(fd))
        snaps = score_fleet(raw_f, fd, MODELS).merge(truth, on="unit")
        old, new = agent_view(snaps, False), agent_view(snaps, True)
        acc.append(snaps.assign(old_sev=old["severity"].to_numpy(), new_sev=new["severity"].to_numpy(),
                                review=new["needs_human_review"].to_numpy(),
                                right_sensor=[faulty.get(u) == s for u, s in zip(snaps["unit"], snaps["sensor_suspect"])]))
    a = pd.concat(acc, ignore_index=True)
    flagged = a["sensor_suspect"] != ""
    rec = np.where(flagged, a["rul_backup"], a["pred_RUL"])
    crit = a["true_RUL"] <= 20
    rows.append({
        "scenario": kind, "snapshots": len(a),
        "flagged_%": round(100 * flagged.mean(), 1),
        "right_sensor_%": round(100 * a.loc[flagged, "right_sensor"].mean(), 1) if (kind != "clean" and flagged.any()) else np.nan,
        "RMSE_model": round(float(np.sqrt(np.mean((a["pred_RUL"] - a["true_RUL"]) ** 2))), 2),
        "RMSE_with_backup": round(float(np.sqrt(np.mean((rec - a["true_RUL"]) ** 2))), 2),
        "critical_caught_before_%": round(100 * (a.loc[crit, "old_sev"] == "CRITICAL").mean(), 1),
        "critical_caught_after_%": round(100 * (a.loc[crit, "new_sev"] == "CRITICAL").mean(), 1),
        "false_CRITICAL_after": int(((a["true_RUL"] > 40) & (a["new_sev"] == "CRITICAL")).sum()),
        "human_review_%": round(100 * a["review"].mean(), 1),
    })
    print(kind, "done", flush=True)

sensor_eval = pd.DataFrame(rows).set_index("scenario")
sensor_eval.to_csv(PROJECT / "reports" / "sensor_fault_evaluation.csv")
print(f"Gate engines only (never trained on), subsets: {SUBSETS_EVAL}")
sensor_eval

C:\Users\hp\Documents\teliport_prototype\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


clean done
bias_up done
bias_down done
stuck done
drift done
Gate engines only (never trained on), subsets: ['FD001', 'FD002', 'FD003', 'FD004']


,snapshots,flagged_%,right_sensor_%,RMSE_model,RMSE_with_backup,critical_caught_before_%,critical_caught_after_%,false_CRITICAL_after,human_review_%
scenario,,,,,,,,,
clean,514,0.0,NaN,9.33,9.33,98.1,98.1,0,0.4
bias_up,514,99.6,99.6,10.59,9.46,96.6,98.5,1,99.8
bias_down,514,99.2,99.8,15.45,9.40,73.8,97.6,0,99.4
stuck,514,55.1,98.9,13.14,10.42,87.9,96.1,0,56.2
drift,514,76.7,99.5,10.62,9.54,96.6,97.6,1,79.8


In [3]:
raw = load_subset("FD001", DATA_DIR)[1].drop(columns=["subset"])
demo = raw[raw["unit"] <= 10].copy()
demo_f, faulty = inject(demo, "FD001", "bias_down", size=3.0, seed=7)        # one broken sensor per engine
path = PROJECT / "data" / "incoming" / "FD001_demo_faulty_sensor.csv"
path.parent.mkdir(parents=True, exist_ok=True)
demo_f.to_csv(path, index=False)

clean = score_fleet(demo, "FD001", MODELS)[["unit", "pred_RUL"]].rename(columns={"pred_RUL": "RUL_clean_data"})
broken = score_fleet(demo_f, "FD001", MODELS)
out = broken[["unit", "pred_RUL", "sensor_suspect", "sensor_fault_score", "rul_backup"]].merge(clean, on="unit")
out.insert(1, "sensor_we_broke", out["unit"].map(faulty))
print("Demo file for the dashboard upload:", path)
out.rename(columns={"pred_RUL": "RUL_broken_data", "rul_backup": "RUL_backup_(sensor_replaced)"})

Demo file for the dashboard upload: C:\Users\hp\Documents\teliport_prototype\data\incoming\FD001_demo_faulty_sensor.csv


,unit,sensor_we_broke,RUL_broken_data,sensor_suspect,sensor_fault_score,RUL_backup_(sensor_replaced),RUL_clean_data
0,1,s21,116.5,s21,15.8,119.4,119.3
1,2,s13,117.7,s13,17.6,118.1,116.8
2,3,s14,48.5,s14,35.1,47.9,48.7
3,4,s20,74.3,s20,16.3,75.8,75.2
4,5,s13,97.5,s13,17.4,91.2,89.4
5,6,s15,98.2,s15,15.6,102.6,103.0
6,7,s17,117.0,s17,12.0,110.4,110.4
7,8,s7,90.5,s7,17.9,92.6,92.7
8,9,s2,118.6,s2,15.8,118.4,118.5
9,10,s8,99.6,s8,16.9,95.6,95.5
